<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week5_instructor_online.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 – Instructor demo: hotels revisited

**For live use while delivering the Week 5 online session, not a student handout.** Same cells as the student walkthrough, with every task answered, plus extras for the Q&A.

Reproduces the textbook's Ch. 8 table exactly (0-mile hotel set to 0.05 before logging).

**Before you start:**

- Go to **File → Save a copy in Drive**. Until you do this, your work only exists in this browser tab – you can't save changes back to GitHub, and closing the tab or losing the session will lose your work.
- Turn off Colab's autocomplete: **Tools → Settings → Editor → uncheck "Show context-powered code completions."** It's a setting on your own Google account, not something built into this notebook, so you'll need to do it once per account.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess

data_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/hotels_vienna.csv"
hotels = pd.read_csv(data_path)

# Same sample as Week 4: hotels in Vienna, 3-4 stars, no price above 600 euros
hotels = hotels[(hotels['accommodation_type'] == 'Hotel') &
                (hotels['city_actual'] == 'Vienna') &
                (hotels['stars'] >= 3) & (hotels['stars'] <= 4) &
                (hotels['price'] <= 600)].copy()
len(hotels)

# Part 1: what shape is the line?

## Step 1: check the distributions

In [ ]:
hotels['ln_price'] = np.log(hotels['price'])
print(f"Skew of price:     {hotels['price'].skew():.2f}")
print(f"Skew of ln(price): {hotels['ln_price'].skew():.2f}")
print(f"Share of hotels within 1 mile: {(hotels['distance'] < 1).mean():.0%}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(hotels['price'], bins=20, color='#457b9d', edgecolor='white')
axes[0].set(xlabel='Price per night (€)', ylabel='Hotels')
axes[1].hist(hotels['ln_price'], bins=20, color='#2d6a4f', edgecolor='white')
axes[1].set(xlabel='ln(price)', ylabel='Hotels')
plt.tight_layout(); plt.show()

## Step 2: look at the shape

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, y in zip(axes, ['price', 'ln_price']):
    smooth = lowess(hotels[y], hotels['distance'], frac=0.3)
    ax.scatter(hotels['distance'], hotels[y], s=12, alpha=0.4, color='#457b9d')
    ax.plot(smooth[:, 0], smooth[:, 1], color='#e63946', linewidth=3)
    ax.set(xlabel='Distance (miles)', ylabel=y)
plt.tight_layout(); plt.show()

## Logging distance: one hotel is 0 miles out

You can't take the log of zero, so – like the textbook – we set that hotel's distance to 0.05 miles first.

In [ ]:
print(f"Hotels at 0 miles: {(hotels['distance'] == 0).sum()}")
hotels['dist_for_log'] = hotels['distance'].replace(0, 0.05)
hotels['ln_dist'] = np.log(hotels['dist_for_log'])

## Four models, one table

In [ ]:
m1 = smf.ols('price ~ distance', data=hotels).fit()
m2 = smf.ols('ln_price ~ distance', data=hotels).fit()
m3 = smf.ols('price ~ ln_dist', data=hotels).fit()

pd.DataFrame({'model': ['level-level', 'log-level', 'level-log'],
              'constant': [m.params.iloc[0] for m in (m1, m2, m3)],
              'slope':    [m.params.iloc[1] for m in (m1, m2, m3)],
              'R2':       [m.rsquared for m in (m1, m2, m3)]}).round(3)

## Task 1: fit the log–log model yourself

Fit `ln_price ~ ln_dist` and put the slope into one sentence about Vienna hotels. Why does the "10% further out" wording suit this model better than "1 mile further out"?

In [ ]:
m4 = smf.ols('ln_price ~ ln_dist', data=hotels).fit()
print(m4.params.round(3), f"R2 = {m4.rsquared:.3f}")
# slope -0.22: hotels 10% further out are about 2.2% cheaper, on average

## Task 2: was last week's straight line defensible?

Work through the roadmap for Week 4's `price ~ distance` model. Write your answers in the cell below:

1. What do the distributions of price and distance look like?
2. What shape did LOWESS show?
3. What transformation would you choose, and why?
4. If you changed the model, would the best deals change?

**Answer.** (1) Price is right-skewed (skew 2.06); 39% of hotels are within a mile of the centre. (2) A steep fall in the first mile, then nearly flat. (3) A log of distance, or a spline with a knot near 1 mile. (4) Yes – see Part 2: four of the five change.

# Part 2: splines

## Fitting one in Python

`beyond_1` is the number of miles beyond the knot – zero for hotels within 1 mile. Its coefficient is the **change** in slope after the knot, not the slope itself.

In [ ]:
hotels['beyond_1'] = np.maximum(hotels['distance'] - 1, 0)
spline = smf.ols('price ~ distance + beyond_1', data=hotels).fit()
print(spline.params.round(2))
print(f"Slope within 1 mile: {spline.params['distance']:.1f}")
print(f"Slope beyond 1 mile: {spline.params['distance'] + spline.params['beyond_1']:.1f}")
print(f"R2: straight line {m1.rsquared:.3f}, spline {spline.rsquared:.3f}")

In [ ]:
xs = np.linspace(0, hotels['distance'].max(), 100)
pred = spline.predict(pd.DataFrame({'distance': xs, 'beyond_1': np.maximum(xs - 1, 0)}))
plt.figure(figsize=(9, 4))
plt.scatter(hotels['distance'], hotels['price'], s=12, alpha=0.4, color='#457b9d')
plt.plot(xs, m1.params.iloc[0] + m1.params.iloc[1] * xs, '--', color='grey', label='Straight line')
plt.plot(xs, pred, color='#e63946', linewidth=3, label='Spline, knot at 1 mile')
plt.xlabel('Distance (miles)'); plt.ylabel('Price per night (€)'); plt.legend()
plt.tight_layout(); plt.show()

## Did it change last week's best deals?

In [ ]:
hotels['res_line'] = m1.resid
hotels['res_spline'] = spline.resid
cols = ['hotel_id', 'distance', 'price']
print("Straight line:"); print(hotels.nsmallest(5, 'res_line')[cols + ['res_line']].round(1))
print("\nSpline:");       print(hotels.nsmallest(5, 'res_spline')[cols + ['res_spline']].round(1))

## Task 3: move the knot

Refit the spline with the knot at **2 miles** instead of 1. Then try **two knots**, at 1 and 4 miles (you'll need a second "beyond" variable). Compare R² each time. Where would you put the knot, and why?

In [ ]:
hotels['beyond_2'] = np.maximum(hotels['distance'] - 2, 0)
hotels['beyond_4'] = np.maximum(hotels['distance'] - 4, 0)
k2 = smf.ols('price ~ distance + beyond_2', data=hotels).fit()
k14 = smf.ols('price ~ distance + beyond_1 + beyond_4', data=hotels).fit()
print(f"knot at 1: {spline.rsquared:.3f}   knot at 2: {k2.rsquared:.3f}   knots at 1 and 4: {k14.rsquared:.3f}")

# Part 3: measurement error, using the ratings

## The real comparison

Ratings averaged over few reviews are noisier. Compare the slope of ln(price) on rating for hotels with fewer than 100 reviews and more than 200.

In [ ]:
rated = hotels.dropna(subset=['rating'])
few = rated[rated['rating_count'] < 100]
many = rated[rated['rating_count'] > 200].copy()

for name, group in [('few reviews', few), ('many reviews', many)]:
    m = smf.ols('ln_price ~ rating', data=group).fit(cov_type='HC1')
    lo, hi = m.conf_int().loc['rating']
    print(f"{name:13s} n={len(group):3d}  avg reviews={group['rating_count'].mean():5.0f}  "
          f"slope={m.params['rating']:.3f}  SE={m.bse['rating']:.3f}  CI=[{lo:.2f}, {hi:.2f}]")

## Task 4: how much noise would it take?

Take the hotels with **more than 200 reviews** (`many`) and add random noise to their ratings. Repeat many times and average the slopes. How big does the noise SD need to be before their slope looks like the few-reviews slope? Check against the attenuation formula, $\beta^* \times Var[x] / (Var[x] + SD^2)$.

```python
rng = np.random.default_rng()
df['x_noisy'] = df['x'] + rng.normal(0, sd, len(df))   # noise with standard deviation sd
```

In [ ]:
rng = np.random.default_rng(5)
b_true = smf.ols('ln_price ~ rating', data=many).fit().params['rating']
var_x = many['rating'].var()
for sd in [0.1, 0.2, 0.3, 0.5]:
    slopes = []
    for _ in range(500):
        many['noisy'] = many['rating'] + rng.normal(0, sd, len(many))
        slopes.append(smf.ols('ln_price ~ noisy', data=many).fit().params['noisy'])
    print(f"SD {sd}: average slope {np.mean(slopes):.3f}   formula {b_true * var_x / (var_x + sd**2):.3f}")

## Extras for the Q&A

Not on the slides – useful if questions go this way.

In [ ]:
# Q1: within-pair comparisons only (same y): level-level vs level-log, log-level vs log-log
print(f"price:     level {m1.rsquared:.3f}  vs  log distance {m3.rsquared:.3f}")
print(f"ln(price): level {m2.rsquared:.3f}  vs  log distance {m4.rsquared:.3f}")

In [ ]:
# Q3: what the best-deal hotels look like under each model
for col in ['res_line', 'res_spline']:
    best = hotels.nsmallest(5, col)
    print(col, f"mean rating {best['rating'].mean():.2f}, mean stars {best['stars'].mean():.2f}, mean distance {best['distance'].mean():.2f}")
print(f"all hotels: mean rating {hotels['rating'].mean():.2f}, mean stars {hotels['stars'].mean():.2f}")

In [ ]:
# Q4: is 0.2 rating points of noise plausible? If each review's score has SD s,
# an average of n reviews has noise SD s / sqrt(n). With s around 1 point:
for n in [20, 57, 417]:
    print(f"n = {n:3d} reviews -> noise SD of the average ~ {1 / np.sqrt(n):.2f}")

In [ ]:
# Q5: is the gap between the two slopes significant? Pool and interact.
rated2 = rated[(rated['rating_count'] < 100) | (rated['rating_count'] > 200)].copy()
rated2['many'] = (rated2['rating_count'] > 200).astype(int)
m = smf.ols('ln_price ~ rating * many', data=rated2).fit(cov_type='HC1')
print(m.params.round(3)); print(f"p-value on the difference: {m.pvalues['rating:many']:.3f}")